In [ ]:
import os
import torch
from tqdm.auto import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np

In [ ]:
class ConvBlock(nn.Module):
    """Double conv: (Conv → BN → ReLU) × 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)
class AttentionGate(nn.Module):
    """
    Attention Gate as in Oktay et al. 2018
    g: gating signal from decoder (coarser, deeper)
    x: skip connection from encoder (finer)
    F_int: intermediate channel size (usually in_ch // 2)
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
    def forward(self, g, x):
        if g.shape[2:] != x.shape[2:]:
            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=True)
        g1  = self.W_g(g)
        x1  = self.W_x(x)
        att = self.psi(F.relu(g1 + x1, inplace=True))
        return x * att
class UpBlock(nn.Module):
    """Upsample → AttentionGate on skip → concat → ConvBlock"""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.att  = AttentionGate(F_g=in_ch // 2, F_l=skip_ch, F_int=skip_ch // 2)
        self.conv = ConvBlock(in_ch // 2 + skip_ch, out_ch)
    def forward(self, x, skip):
        x    = self.up(x)
        skip = self.att(g=x, x=skip)
        x    = torch.cat([skip, x], dim=1)
        return self.conv(x)

In [ ]:
class AttentionUNet(nn.Module):
    """
    Full Attention U-Net for binary segmentation.
    Input : (B, in_channels, H, W)   — use in_channels=1 for grayscale MRI
    Output: (B, 1, H, W)             — raw logits, apply sigmoid for prob
    """
    def __init__(self, in_channels=1, base_ch=64):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_ch)
        self.enc2 = ConvBlock(base_ch,     base_ch * 2)
        self.enc3 = ConvBlock(base_ch * 2, base_ch * 4)
        self.enc4 = ConvBlock(base_ch * 4, base_ch * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base_ch * 8, base_ch * 16)
        self.dec4 = UpBlock(base_ch * 16, base_ch * 8,  base_ch * 8)
        self.dec3 = UpBlock(base_ch * 8,  base_ch * 4,  base_ch * 4)
        self.dec2 = UpBlock(base_ch * 4,  base_ch * 2,  base_ch * 2)
        self.dec1 = UpBlock(base_ch * 2,  base_ch,      base_ch)
        self.out_conv = nn.Conv2d(base_ch, 1, kernel_size=1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(b,  e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.out_conv(d1)

In [ ]:
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=1.33, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        TP = (probs * targets).sum(dim=(2, 3))
        FP = ((1 - targets) * probs).sum(dim=(2, 3))
        FN = (targets * (1 - probs)).sum(dim=(2, 3))
        Tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        FT_loss = (1 - Tversky) ** self.gamma
        return FT_loss.mean()
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num   = 2 * (probs * targets).sum(dim=(2, 3))
        den   = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + self.smooth
        return 1 - (num / den).mean()
class CombinedLoss(nn.Module):
    """Dice + BCE — same as Attention U-Net pipeline"""
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.bce  = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.bce(logits, targets)

In [ ]:
class BRISCDataset(Dataset):
    """
    Expects directory structure:
        root/
          images/  *.jpg
          masks/   *.png   (same basename as images)
    """
    def __init__(self, root, img_size=256):
        self.img_dir  = os.path.join(root, "images")
        self.msk_dir  = os.path.join(root, "masks")
        self.names    = sorted(os.listdir(self.img_dir))
        self.img_size = img_size
        self.img_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
        self.msk_tf = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=Image.NEAREST),
            transforms.ToTensor(),
        ])
    def __len__(self):
        return len(self.names)
    def __getitem__(self, idx):
        name     = self.names[idx]
        img_path = os.path.join(self.img_dir, name)
        msk_path = os.path.join(self.msk_dir, os.path.splitext(name)[0] + ".png")
        img  = Image.open(img_path).convert("L")
        mask = Image.open(msk_path).convert("L")
        img  = self.img_tf(img)
        mask = self.msk_tf(mask)
        mask = (mask > 0.5).float()
        return img, mask

In [ ]:
def dice_score(logits, targets, threshold=0.5, smooth=1e-6):
    probs  = (torch.sigmoid(logits) > threshold).float()
    num    = 2 * (probs * targets).sum(dim=(2, 3))
    den    = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + smooth
    return (num / den).mean().item()
def iou_score(logits, targets, threshold=0.5, smooth=1e-6):
    probs  = (torch.sigmoid(logits) > threshold).float()
    inter  = (probs * targets).sum(dim=(2, 3))
    union  = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) - inter + smooth
    return (inter / union).mean().item()

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="Training", leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    return total_loss / len(loader)
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    pbar = tqdm(loader, desc="Validating", leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, masks)
        total_loss += loss.item()
        total_dice += dice_score(logits, masks)
        total_iou  += iou_score(logits, masks)
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

In [ ]:
DATA_ROOT         = "/home/kartik/Desktop/shalini/brain_brisc/brisc_processed"
SAVE_PATH         = "best_attention_unet.pth"
RESUME_CHECKPOINT = "last_checkpoint.pth"
IMG_SIZE          = 256
BATCH_SIZE        = 12
NUM_EPOCHS        = 10
LR                = 1e-4
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
train_dir = os.path.join(DATA_ROOT, "train")
val_dir = os.path.join(DATA_ROOT, "val")
train_ds = BRISCDataset(train_dir, img_size=IMG_SIZE)
val_ds = BRISCDataset(val_dir, img_size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
model     = AttentionUNet(in_channels=1, base_ch=64).to(DEVICE)
criterion = CombinedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler    = torch.cuda.amp.GradScaler()
best_val_dice = 0.0
start_epoch   = 0
if os.path.exists(RESUME_CHECKPOINT):
    print(f"Loading checkpoint from {RESUME_CHECKPOINT}...")
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    if "scaler_state" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state"])
    start_epoch = checkpoint["epoch"]
    best_val_dice = checkpoint.get("best_val_dice", 0.0)
    print(f"Resuming from epoch {start_epoch} with Best Val Dice: {best_val_dice:.4f}")
else:
    print("Starting fresh training...")
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=999)
if os.path.exists(RESUME_CHECKPOINT) and "scheduler_state" in checkpoint:
    scheduler.load_state_dict(checkpoint["scheduler_state"])
for epoch in range(start_epoch + 1, start_epoch + NUM_EPOCHS + 1):
    trn_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
    val_loss, val_dice, val_iou = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    print(
        f"Epoch [{epoch:03d}]  "
        f"Train Loss: {trn_loss:.4f}  |  "
        f"Val Loss: {val_loss:.4f}  |  "
        f"Val Dice: {val_dice:.4f}  |  "
        f"Val IoU: {val_iou:.4f}"
    )
    torch.save({
        "epoch":          epoch,
        "model_state":    model.state_dict(),
        "optimizer_state":optimizer.state_dict(),
        "scheduler_state":scheduler.state_dict(),
        "scaler_state":   scaler.state_dict(),
        "best_val_dice":  best_val_dice,
    }, RESUME_CHECKPOINT)
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save({
            "epoch":          epoch,
            "model_state":    model.state_dict(),
            "optimizer_state":optimizer.state_dict(),
            "val_dice":       val_dice,
            "val_iou":        val_iou,
            "val_loss":       val_loss,
        }, SAVE_PATH)
        print(f"  ✓ Best model saved  (Dice: {best_val_dice:.4f})")
print(f"\nTraining run complete. Best Val Dice overall: {best_val_dice:.4f}")
print(f"Latest checkpoint saved to: {RESUME_CHECKPOINT}")
print(f"Best model saved to: {SAVE_PATH}")